# Real-Time BCI Simulation via Lab Streaming Layer (LSL) - DEMO

## 1. Objective
This notebook demonstrates the deployment of the trained EEGNet model in a simulated real-time environment. We utilize the **Lab Streaming Layer (LSL)** protocol to establish a communication bridge between a data source (Mock EEG Headset) and a signal processor (BCI Decoder).

## 2. System Architecture
The pipeline consists of two primary components:
1. **The Mock streamer:** This component streams pre-recorded EEG data from the BCI Competition IV 2a dataset at the native sampling rate (250 Hz), simulating a live wearable device.
2. **The Real-time classifier:** This component "listens" to the LSL stream, buffers incoming samples into temporal windows, and applies the trained `.keras` model to predict user intent in real-time.



## 3. Technical Requirements
* **pylsl:** For cross-process data streaming.
* **Pre-trained model:** `final_eegnet_bci_model.keras` generated in Notebook 05.
* **Window size:** 3.0 seconds (751 samples) as per training specifications.

## Dependencies and model loading

In [1]:
import numpy as np
import time
import mne
import os
import tensorflow as tf
from pylsl import StreamInfo, StreamOutlet, StreamInlet, resolve_byprop

# Load the optimized model for real-time inference
MODEL_PATH = 'final_eegnet_bci_model.keras'
model = tf.keras.models.load_model(MODEL_PATH)

# Global variables
SFREQ = 250.0  # Sampling frequency
CHANNELS = 22  # EEG Channels
WINDOW_SIZE = 751  # Number of samples for one prediction (3 seconds)

## Mock streamer

In [2]:
import threading
import time
from pylsl import StreamInfo, StreamOutlet

def start_smart_streamer(subject_id='A07T'):
    """
    Simulates a live EEG headset with an automatic stop condition.
    """
    def streamer_loop():
        # Load the cleaned subject data (filtered 1-40 Hz)
        file_path = f'./cleaned_data_1_40/{subject_id}_clean-raw_1_40.fif'
        raw = mne.io.read_raw_fif(file_path, preload=True, verbose=False)
        raw.pick(['eeg'])
        data = raw.get_data()
        
        # Setup LSL Stream
        info = StreamInfo('MockEEG', 'EEG', CHANNELS, SFREQ, 'float32', 'bci_uid_001')
        outlet = StreamOutlet(info)
        
        print(f"[STREAMER] Session started for {subject_id}. Streaming {data.shape[1]} samples.")
        
        for sample_idx in range(data.shape[1]):
            # Check if external stop signal was sent
            if not getattr(threading.current_thread(), "do_run", True):
                print("[STREAMER] Manual stop received.")
                break
                
            sample = data[:, sample_idx].tolist()
            outlet.push_sample(sample)
            time.sleep(1.0 / SFREQ)
            
        print("[STREAMER] Data exhausted. Session finished.")

    t = threading.Thread(target=streamer_loop)
    t.do_run = True
    t.start()
    return t

In [ ]:
# start the smart streamer for subject A07T
stream_thread = start_smart_streamer('A07T')

C:\Users\ACER\AppData\Local\Temp\ipykernel_17864\3635274720.py:12: RuntimeWarning: This filename (./cleaned_data_1_40/A07T_clean-raw_1_40.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(file_path, preload=True, verbose=False)


[STREAMER] Session started for A07T. Streaming 680271 samples.
[STREAMER] Manual stop received.
[STREAMER] Data exhausted. Session finished.


## Real-time classifier

In [ ]:
def run_bci_decoder():
    """
    Connects to the LSL stream and predicts motor imagery intent in real-time.
    """
    # 1. Resolve the EEG stream
    print("Looking for an EEG stream...")
    streams = resolve_byprop('type', 'EEG', timeout=5)
    
    if not streams:
        print("No LSL stream found. Please start the Mock Streamer first.")
        return
        
    inlet = StreamInlet(streams[0])
    print("Connected to MockEEG stream.")
    
    # 2. Setup circular buffer for the sliding window
    buffer = np.zeros((CHANNELS, WINDOW_SIZE))
    class_names = ['Left Hand', 'Right Hand', 'Foot', 'Tongue']
    
    print("\nStarting real-time classification loop...")
    try:
        while True:
            # Pull a sample from LSL
            sample, timestamp = inlet.pull_sample()
            
            # Update circular buffer (Shift left and add new sample to the right)
            buffer = np.roll(buffer, -1, axis=1)
            buffer[:, -1] = sample
            
            # For demonstration, we perform inference every 0.5 seconds (125 samples)
            # to avoid overloading the CPU, even if the buffer is 3 seconds long.
            if timestamp % 0.5 < (1.0 / SFREQ):
                
                # --- PRE-PROCESSING ---
                # 1. Trial Standardization (Z-score)
                input_data = (buffer - np.mean(buffer, axis=-1, keepdims=True)) / np.std(buffer, axis=-1, keepdims=True)
                
                # 2. Reshape for EEGNet (1, Channels, Samples, 1)
                input_data = input_data.reshape(1, CHANNELS, WINDOW_SIZE, 1)
                
                # --- INFERENCE ---
                prediction = model.predict(input_data, verbose=0)
                pred_class = np.argmax(prediction)
                confidence = np.max(prediction)
                
                # --- FEEDBACK ---
                if confidence > 0.60: # Threshold to filter out low-confidence noise
                    print(f"Predicted Command: {class_names[pred_class]} | Confidence: {confidence:.2%}")
                else:
                    print("Status: Neutral / Searching...")
                    
    except KeyboardInterrupt:
        print("Inference loop stopped.")

# run_bci_decoder()

### During a live demonstration
 It is recommended to **stop the process manually after observing a few successful predictions**. 
For full-session validation, the smart streamer will automatically terminate once the subject's dataset (approx. 40 minutes of EEG) is exhausted.

In [5]:
run_bci_decoder()


Looking for an EEG stream...
Connected to MockEEG stream.

Starting real-time classification loop...
Predicted Command: Tongue | Confidence: 83.41%
Predicted Command: Tongue | Confidence: 63.96%
Predicted Command: Tongue | Confidence: 74.54%
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Status: Neutral / Searching...
Predicted Command: Tongue | Confidence: 62.48%
Status: Neutral / Searching...
Predicted Command: Tongue | Confidence: 69.37%
Predicted Command: Tongue | Confidence: 86.40%
Predicted Command: Right Hand | Confidence: 81.49%
Predicted Command: Right Hand | Confidence: 63.75%
Status: Neutral / Searching...
Status: Neutral / S

In [6]:
# STOP SIGNAL
if 'stream_thread' in locals():
    stream_thread.do_run = False
    print("Streamer shutdown signal sent.")
else:
    print("No active streamer found.")

Streamer shutdown signal sent.


## Real-Time dashboard execution

To ensure high performance and avoid UI blocking within the Jupyter environment, the visual interface has been migrated to a standalone **Streamlit** application. 

The system follows a distributed architecture:
1. **The notebook (engine):** Handles the LSL Mock Streamer and the background data flow.
2. **The external script (dashboard):** A dedicated Python script (`7.bci_demo.py`) that connects to the stream and renders the predictions.

### How to run the Dashboard:
1. **Start the streamer:** execute the cell above (`start_smart_streamer`) to begin pushing EEG data into the LSL network.
2. **Launch Streamlit:** open your terminal or command prompt in the project directory and run:
   ```bash
   streamlit run 7.bci._demo.py

### How to Stop the System:
1. **Stop the Dashboard:** go to the terminal where Streamlit is running and press Ctrl + C. You can then close the browser tab.

2. **Stop the Engine (LSL):** Return to this notebook and execute the cleanup cell below:
stream_thread.do_run = False
This is crucial to prevent zombie LSL streams and free up CPU resources.
3. Automatic Stop: The start_smart_streamer is designed with a sample-limit exit condition; it will automatically terminate once the subject's dataset is fully processed (approx. 40 minutes of EEG).

##  Final Considerations
This simulation demonstrates the transition from a static deep learning model to a dynamic BCI application. By utilizing **LSL**, the system architecture remains identical whether the input is a pre-recorded file or a high-end medical EEG headset.

**Key achievements of this project:**
* Successfully decoded 4 classes of motor imagery with high stability.
* Implemented a Subject-Independent training approach via pooling.
* Built a modular real-time pipeline compatible with neuro-scientific standards.

**Future Directions:**
* Integration with a robotic GUI or VR environment for visual feedback.
* Implementation of Transfer Learning to adapt the pooled model to new users with minimal data.